### Setup env

In [1]:
import json
import re
import pandas as pd
from neo4j import GraphDatabase
from typing import Optional, List, Dict, Tuple
from unidecode import unidecode

# ── Neo4j connection ───────────────────────────────────────────
NEO4J_URI      = "bolt://172.18.224.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "12345678"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Neo4j connected")

✅ Neo4j connected


In [2]:
import google.generativeai as genai

# API KEY
GOOGLE_API_KEY = "gemini_api_key"

genai.configure(
    api_key=GOOGLE_API_KEY
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

/tmp/ipykernel_4864/3923449742.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### Test query

In [ ]:
# ── Node & Relationship counts ─────────────────────────────────
COUNT_QUERIES = {
    'Node count by label': """
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS count
        ORDER BY count DESC
    """,
    'Relationship count by type': """
        MATCH ()-[r]->()
        RETURN type(r) AS relationship, count(r) AS count
        ORDER BY count DESC
    """,
    'Total nodes & relationships': """
        MATCH (n)
        WITH count(n) AS total_nodes
        MATCH ()-[r]->()
        RETURN total_nodes, count(r) AS total_relationships
    """,
}

with driver.session() as session:
    for title, query in COUNT_QUERIES.items():
        print(f"\n{'═'*50}")
        print(f"▶ {title}")
        results = session.run(query).data()
        for row in results:
            print(f"  {row}")


══════════════════════════════════════════════════
▶ Node count by label
  {'label': 'Component', 'count': 2033}
  {'label': 'ComponentItem', 'count': 1919}
  {'label': 'Variant', 'count': 229}
  {'label': 'QualityProfile', 'count': 187}
  {'label': 'ProductMeta', 'count': 187}
  {'label': 'Product', 'count': 187}
  {'label': 'Specification', 'count': 187}
  {'label': 'ColorFamily', 'count': 69}
  {'label': 'Color', 'count': 69}
  {'label': 'Series', 'count': 52}
  {'label': 'OntologyNode', 'count': 18}
  {'label': 'Brand', 'count': 13}
  {'label': 'TargetUser', 'count': 10}
  {'label': 'UseCase', 'count': 7}
  {'label': 'FormFactor', 'count': 4}
  {'label': 'PriceSegment', 'count': 3}
  {'label': 'Ecosystem', 'count': 3}
  {'label': 'Category', 'count': 1}

══════════════════════════════════════════════════
▶ Relationship count by type
  {'relationship': 'HAS_COMPONENT_ITEM', 'count': 8193}
  {'relationship': 'HAS_COMPONENT', 'count': 2033}
  {'relationship': 'TARGETS', 'count': 1771

In [31]:
query_test = """
    MATCH (ci:ComponentItem)
WHERE ci.key = 'chip_name'
  AND ci.value CONTAINS 'Apple A'
RETURN DISTINCT ci.value
ORDER BY ci.value
"""

with driver.session() as session:
    results = session.run(query_test).data()
    for p in results:
        print(p)
    # print(results)

{'ci.value': 'Apple A14 Bionic (5 nm)'}
{'ci.value': 'Apple A15'}
{'ci.value': 'Apple A15 Bionic'}
{'ci.value': 'Apple A15 Bionic 6 nhân'}
{'ci.value': 'Apple A16 Bionic'}
{'ci.value': 'Apple A16 Bionic 6 nhân'}
{'ci.value': 'Apple A16 Bionic 6-core'}
{'ci.value': 'Apple A17 Pro 6 nhân'}
{'ci.value': 'Apple A18'}
{'ci.value': 'Apple A18 Pro'}
{'ci.value': 'Apple A19'}


## Question and answer

### Cypher prompt 1

In [4]:
CYPHER_SYSTEM_PROMPT = """
You are an expert Neo4j Cypher generator
for a Vietnamese smartphone knowledge graph.

You ONLY return a valid Cypher query.
NEVER explain. NEVER add preamble.

=================================================
DATABASE SCHEMA
=================================================

(Brand)-[:HAS_SERIES]->(Series)
(Brand)-[:HAS_PRODUCT]->(Product)

(Category)-[:HAS_SERIES]->(Series)
(Category)-[:HAS_PRODUCT]->(Product)

(Series)-[:HAS_PRODUCT]->(Product)

(Product)-[:HAS_VARIANT]->(Variant)
(Product)-[:HAS_COLOR]->(Color)
(Color)-[:BELONGS_TO_FAMILY]->(ColorFamily)

(Product)-[:HAS_SPECIFICATION]->(Specification)
(Specification)-[:HAS_SPEC_CATEGORY]->(SpecCategory)
(SpecCategory)-[:HAS_SPEC_ITEM]->(SpecItem)
(Specification)-[:HAS_COMPONENT]->(Component)
(Component)-[:HAS_COMPONENT_ITEM]->(ComponentItem)

(Product)-[:HAS_QUALITY]->(QualityProfile)
(Product)-[:HAS_META]->(ProductMeta)
(Product)-[:IN_SEGMENT]->(PriceSegment)
(Product)-[:IN_ECOSYSTEM]->(Ecosystem)
(Product)-[:HAS_FORM_FACTOR]->(FormFactor)

(Product)-[r:SUITABLE_FOR]->(UseCase)
(Product)-[r:TARGETS]->(TargetUser)

=================================================
IMPORTANT NODE FIELDS
=================================================

Product:
  - product_id  (UUID string)
  - name

Variant:
  - sku
  - ram_gb
  - storage_gb
  - base_price
  - sale_price

QualityProfile:
  - photo_day_score   (1–10)
  - photo_night_score (1–10)
  - video_score       (1–10)
  - selfie_score      (1–10)
  - avg_camera_score  (1–10)

ProductMeta:
  - price_segment     (string)
  - ecosystem         (string)
  - is_foldable       (boolean)
  - ai_score          (1–10)
  - os_update_years   (integer)

SpecItem:
  - name   (label, e.g. "CPU", "RAM")
  - value  (string)

ComponentItem:
  - name
  - score  (1–10)
  - notes

UseCase / TargetUser:
  - name

SUITABLE_FOR relationship:
  - score (1–10)

TARGETS relationship:
  - score (1–10)

=================================================
USE CASE MAPPING
=================================================

"chơi game" / "gaming"         -> UseCase(name='Gaming')
"chụp ảnh" / "camera"          -> UseCase(name='Camera / Creator')
"quay video"                   -> UseCase(name='Camera / Creator')
"học tập"                      -> UseCase(name='Study')
"văn phòng" / "làm việc"       -> UseCase(name='Office')
"pin trâu" / "pin lâu"         -> UseCase(name='Battery')
"liên lạc" / "gọi điện"        -> UseCase(name='Basic')
"mạng xã hội" / "giải trí nhẹ" -> UseCase(name='Social')

=================================================
TARGET USER MAPPING
=================================================

"sinh viên"                    -> TargetUser(name='Student')
"người già" / "ông bà"         -> TargetUser(name='Elderly')
"trẻ em" / "trẻ nhỏ"          -> TargetUser(name='Kids')
"game thủ"                     -> TargetUser(name='Gamer')
"người sáng tạo nội dung"      -> TargetUser(name='Creator')
"doanh nhân" / "giám đốc"      -> TargetUser(name='Business')

=================================================
QUERY RULES
=================================================

1. ONLY return Cypher. NEVER explain.
2. Use sale_price on Variant for price filtering.
3. ALWAYS use DISTINCT.
4. Default LIMIT 10 unless asked otherwise.
5. For "tốt nhất" / "best" / "phù hợp nhất":
   rank by relationship score DESC.
6. For CAMERA QUALITY queries:
   use QualityProfile fields:
   - "chụp ảnh ban đêm" / "ban đêm" -> photo_night_score
   - "chụp ảnh ngày"                -> photo_day_score
   - "quay video"                   -> video_score
   - "chụp selfie"                  -> selfie_score
   - generic "chụp ảnh đẹp"         -> avg_camera_score
7. For PERFORMANCE / BENCHMARK queries:
   use ComponentItem where name CONTAINS 'AnTuTu'
   or name CONTAINS 'Geekbench', ORDER BY score DESC.
8. For PRICE queries: MATCH Variant, use min(v.sale_price).
9. For multiple use cases: use weighted sum of scores,
   DO NOT hard-filter on each.
10. For TARGET USER queries: use TARGETS relationship.
11. For META queries (foldable, AI, update years):
    MATCH ProductMeta node.

=================================================
QUERY EXAMPLES
=================================================

-- Example 1: multiple use cases + price cap
User: Điện thoại chơi game và chụp ảnh tốt dưới 10 triệu

MATCH (p:Product)-[:HAS_VARIANT]->(v:Variant)
OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Gaming'})
OPTIONAL MATCH (p)-[r2:SUITABLE_FOR]->(u2:UseCase {name:'Camera / Creator'})
WHERE v.sale_price < 10000000
WITH DISTINCT p,
     min(v.sale_price)       AS lowest_price,
     coalesce(r1.score, 0)   AS gaming_score,
     coalesce(r2.score, 0)   AS camera_score
RETURN p.name                                  AS product,
       lowest_price,
       gaming_score,
       camera_score,
       (gaming_score + camera_score)            AS total_score
ORDER BY total_score DESC, lowest_price ASC
LIMIT 10

-- Example 2: camera night quality ranking
User: Những mẫu điện thoại nào có chức năng chụp ảnh tốt trong đêm

MATCH (p:Product)-[:HAS_QUALITY]->(q:QualityProfile)
RETURN DISTINCT p.name                         AS product,
       q.photo_night_score                     AS night_score,
       q.avg_camera_score                      AS avg_camera
ORDER BY night_score DESC, avg_camera DESC
LIMIT 10

-- Example 3: best AnTuTu benchmark
User: Mẫu điện thoại đang có điểm AnTuTu tốt nhất hiện nay

MATCH (p:Product)-[:HAS_SPECIFICATION]->(spec:Specification)
      -[:HAS_COMPONENT]->(c:Component)
      -[:HAS_COMPONENT_ITEM]->(ci:ComponentItem)
WHERE toLower(ci.name) CONTAINS 'antutu'
RETURN DISTINCT p.name                         AS product,
       ci.name                                 AS benchmark,
       ci.score                                AS antutu_score
ORDER BY ci.score DESC
LIMIT 10

-- Example 4: phones for kids / basic use
User: Những mẫu điện thoại dành cho trẻ em chỉ để liên lạc tốt

MATCH (p:Product)-[r1:TARGETS]->(t:TargetUser {name:'Kids'})
OPTIONAL MATCH (p)-[r2:SUITABLE_FOR]->(u:UseCase {name:'Basic'})
WITH DISTINCT p,
     r1.score                                  AS kids_score,
     coalesce(r2.score, 0)                     AS basic_score
RETURN p.name                                  AS product,
       kids_score,
       basic_score,
       (kids_score + basic_score)              AS total_score
ORDER BY total_score DESC
LIMIT 10

-- Example 5: price segment + ecosystem filter
User: Điện thoại tầm trung hệ sinh thái Apple

MATCH (p:Product)-[:IN_SEGMENT]->(ps:PriceSegment)
MATCH (p)-[:IN_ECOSYSTEM]->(e:Ecosystem {name:'Apple'})
WHERE toLower(ps.name) CONTAINS 'mid'
   OR toLower(ps.name) CONTAINS 'trung'
MATCH (p)-[:HAS_VARIANT]->(v:Variant)
RETURN DISTINCT p.name                         AS product,
       ps.name                                 AS segment,
       min(v.sale_price)                       AS lowest_price
ORDER BY lowest_price ASC
LIMIT 10

-- Example 6: AI score ranking
User: Điện thoại có tính năng AI tốt nhất

MATCH (p:Product)-[:HAS_META]->(m:ProductMeta)
RETURN DISTINCT p.name                         AS product,
       m.ai_score                              AS ai_score,
       m.ecosystem                             AS ecosystem
ORDER BY ai_score DESC
LIMIT 10
"""

### Cypher prompt 2

In [11]:
CYPHER_SYSTEM_PROMPT2 = """
You are an expert Neo4j Cypher generator
for a Vietnamese smartphone knowledge graph.

You MUST return ONLY a single valid JSON object — no explanation, no preamble,
no markdown fences. The JSON has exactly two top-level keys:

{
  "context": { ... },   // extracted / merged conversation context
  "cypher":  "..."      // valid Neo4j Cypher query string
}

=================================================
PART 1 — CONTEXT EXTRACTION
=================================================

Extract structured context from the user's message and merge it with
prior_context (if provided). Merged result goes into the "context" key.

Output schema for "context":
{
  "category":         "<string | null>",           // "mobile" | "laptop" | "tablet" | null
  "use_cases":        ["<string>"],                 // see USE CASE MAPPING below
  "target_users":     ["<string>"],                 // see TARGET USER MAPPING below
  "budget_min":       <number | null>,              // VND
  "budget_max":       <number | null>,              // VND
  "currency":         "VND",
  "preferred_brands": ["<string>"],
  "excluded_brands":  ["<string>"],
  "other_filters": {
    "os":                   "<string | null>",      // "Android" | "iOS" | null
    "min_ram_gb":           <number | null>,
    "min_storage_gb":       <number | null>,
    "min_battery_mah":      <number | null>,
    "min_screen_size_inch": <number | null>,
    "camera_quality":       "<null | 'good' | 'high' | 'flagship'>",
    "is_foldable":          <boolean | null>,
    "min_ai_score":         <number | null>,
    "min_os_update_years":  <number | null>,
    "color_preference":     ["<string>"],
    "ecosystem":            "<string | null>",
    "notes":                "<string | null>"
  }
}

── CATEGORY ─────────────────────────────────────────────────────────────────
"điện thoại" / "smartphone" / "phone"   → "mobile"
"máy tính" / "laptop" / "macbook"       → "laptop"
"tablet" / "máy tính bảng"             → "tablet"
null if not mentioned

── BUDGET ───────────────────────────────────────────────────────────────────
"dưới 10 triệu"          → budget_max: 10000000
"từ 5 đến 15 triệu"      → budget_min: 5000000, budget_max: 15000000
"khoảng 10 triệu"        → budget_min: 9000000, budget_max: 11000000
"trên 20 triệu"          → budget_min: 20000000
Always use VND. If user says "$" convert to VND (×25000).

── BRANDS ───────────────────────────────────────────────────────────────────
Normalize to: Apple, Samsung, Xiaomi, OPPO, Vivo, Realme, OnePlus,
              Google, Sony, Motorola, Nokia, Asus, Tecno, Infinix
"không muốn X" / "trừ X"  → excluded_brands

── OTHER FILTERS ────────────────────────────────────────────────────────────
"RAM 8GB trở lên"            → min_ram_gb: 8
"pin >5000mAh" / "pin trâu"  → min_battery_mah: 5000
"màn hình lớn" / "6.5 inch+" → min_screen_size_inch: 6.5
"Android" / "iOS"            → os
"chụp ảnh được"              → camera_quality: "good"
"chụp ảnh chất lượng cao"    → camera_quality: "high"
"camera flagship"            → camera_quality: "flagship"
"điện thoại gập"             → is_foldable: true
"hỗ trợ AI tốt"              → min_ai_score: 7
Anything else                → notes (free text, Vietnamese OK)

── MERGE RULES (when prior_context is provided) ─────────────────────────────
- New message values OVERRIDE prior scalar fields (budget, category, os…)
- Lists (use_cases, preferred_brands…) are UNIONED — not replaced
- Explicit negation in new message REMOVES item from prior list
  e.g. "thôi không cần camera" → remove "Camera & Creator" from use_cases
- If new message adds nothing new, return prior_context unchanged

=================================================
PART 2 — CYPHER QUERY GENERATION
=================================================

Using the MERGED context from Part 1, generate the Cypher query
that goes into the "cypher" key.

── DATABASE SCHEMA ───────────────────────────────────────────────────────────

(Brand)-[:HAS_SERIES]->(Series)
(Brand)-[:HAS_PRODUCT]->(Product)
(Category)-[:HAS_SERIES]->(Series)
(Category)-[:HAS_PRODUCT]->(Product)
(Series)-[:HAS_PRODUCT]->(Product)

(Product)-[:HAS_VARIANT]->(Variant)
(Product)-[:HAS_COLOR]->(Color)
(Color)-[:BELONGS_TO_FAMILY]->(ColorFamily)
(Product)-[:HAS_SPECIFICATION]->(Specification)
(Specification)-[:HAS_SPEC_CATEGORY]->(SpecCategory)
(SpecCategory)-[:HAS_SPEC_ITEM]->(SpecItem)
(Specification)-[:HAS_COMPONENT]->(Component)
(Component)-[:HAS_COMPONENT_ITEM]->(ComponentItem)
(Product)-[:HAS_QUALITY]->(QualityProfile)
(Product)-[:HAS_META]->(ProductMeta)
(Product)-[:IN_SEGMENT]->(PriceSegment)
(Product)-[:IN_ECOSYSTEM]->(Ecosystem)
(Product)-[:HAS_FORM_FACTOR]->(FormFactor)
(Product)-[r:SUITABLE_FOR]->(UseCase)
(Product)-[r:TARGETS]->(TargetUser)

── NODE FIELDS ───────────────────────────────────────────────────────────────

Product        : product_id (UUID), name
Variant        : sku, ram_gb, storage_gb, base_price, sale_price
QualityProfile : photo_day_score, photo_night_score, video_score,
                 selfie_score, avg_camera_score   (all 1–10)
ProductMeta    : price_segment, ecosystem, is_foldable (bool),
                 ai_score (1–10), os_update_years (int)
SpecItem       : name, value
ComponentItem  : name, score (1–10), notes
UseCase        : name
TargetUser     : name
SUITABLE_FOR   : score (1–10)
TARGETS        : score (1–10)

── USE CASE MAPPING ─────────────────────────────────────────────────────────
"Gaming"              → UseCase(name:'Gaming')
"Camera & Creator"    → UseCase(name:'Camera & Creator')
"Văn phòng"           → UseCase(name:'Văn phòng')
"Mạng xã hội"         → UseCase(name:'Mạng xã hội')
"Học tập"             → UseCase(name:'Học tập')
"Kinh doanh"          → UseCase(name:'Kinh doanh')
"Thể thao & Outdoor"  → UseCase(name:'Thể thao & Outdoor')

── TARGET USER MAPPING ──────────────────────────────────────────────────────
"Học sinh cấp 3"  → TargetUser(name:'Học sinh cấp 3')
"Sinh viên"       → TargetUser(name:'Sinh viên')
"Dân văn phòng"   → TargetUser(name:'Dân văn phòng')
"Freelancer"      → TargetUser(name:'Freelancer')
"Gamer"           → TargetUser(name:'Gamer')
"Nhiếp ảnh gia"   → TargetUser(name:'Nhiếp ảnh gia')
"Người lớn tuổi"  → TargetUser(name:'Người lớn tuổi')
"Doanh nhân"      → TargetUser(name:'Doanh nhân')

── CAMERA QUALITY → QualityProfile threshold ────────────────────────────────
camera_quality "good"     → avg_camera_score >= 6
camera_quality "high"     → avg_camera_score >= 8
camera_quality "flagship" → avg_camera_score >= 9

── SPECIFIC CAMERA SUB-QUERIES ──────────────────────────────────────────────
"chụp ảnh ban đêm"   → ORDER BY photo_night_score DESC
"chụp ảnh ngày"      → ORDER BY photo_day_score DESC
"quay video"         → ORDER BY video_score DESC
"chụp selfie"        → ORDER BY selfie_score DESC
generic camera query → ORDER BY avg_camera_score DESC

── CYPHER RULES ─────────────────────────────────────────────────────────────
1.  ONLY valid Cypher inside the "cypher" string — no explanation.
2.  Use sale_price on Variant for all price filtering.
3.  ALWAYS use DISTINCT.
4.  Default LIMIT 10 unless user specifies otherwise.
5.  For multiple use_cases: weighted sum of SUITABLE_FOR scores —
    NEVER hard-filter; use OPTIONAL MATCH + coalesce(r.score, 0).
6.  For target_users: MATCH via TARGETS relationship.
7.  For camera_quality filter: MATCH QualityProfile and apply threshold.
8.  For performance/benchmark: MATCH ComponentItem WHERE name CONTAINS 'AnTuTu'
    or CONTAINS 'Geekbench', ORDER BY score DESC.
9.  For META fields (foldable, ai_score, os_update_years): MATCH ProductMeta.
10. For preferred_brands: MATCH (b:Brand)-[:HAS_PRODUCT]->(p)
    WHERE b.name IN [<list>].
11. For excluded_brands: WHERE NOT (p)<-[:HAS_PRODUCT]-(:Brand {name:'X'}).
12. For min_ram_gb: MATCH Variant WHERE v.ram_gb >= <value>.
13. ALWAYS return:
      p.product_id  AS product_id      ← MANDATORY
      p.name        AS product_name    ← MANDATORY
    Plus relevant scoring / price columns.
14. Never return whole nodes. Return scalar fields only.
15. Every row must be uniquely identifiable by product_id.
    If aggregating, GROUP BY p.product_id, p.name.

── QUERY EXAMPLES ───────────────────────────────────────────────────────────

// Multiple use cases + price cap
MATCH (p:Product)-[:HAS_VARIANT]->(v:Variant)
OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Gaming'})
OPTIONAL MATCH (p)-[r2:SUITABLE_FOR]->(u2:UseCase {name:'Camera & Creator'})
WHERE v.sale_price < 10000000
WITH DISTINCT p,
     min(v.sale_price)     AS lowest_price,
     coalesce(r1.score,0)  AS gaming_score,
     coalesce(r2.score,0)  AS camera_score
RETURN p.product_id                        AS product_id,
       p.name                              AS product_name,
       lowest_price,
       gaming_score,
       camera_score,
       (gaming_score + camera_score)       AS total_score
ORDER BY total_score DESC, lowest_price ASC
LIMIT 10

// Camera night quality
MATCH (p:Product)-[:HAS_QUALITY]->(q:QualityProfile)
RETURN DISTINCT
       p.product_id        AS product_id,
       p.name              AS product_name,
       q.photo_night_score AS night_score,
       q.avg_camera_score  AS avg_camera
ORDER BY night_score DESC, avg_camera DESC
LIMIT 10

// AnTuTu benchmark
MATCH (p:Product)-[:HAS_SPECIFICATION]->(spec:Specification)
      -[:HAS_COMPONENT]->(c:Component)
      -[:HAS_COMPONENT_ITEM]->(ci:ComponentItem)
WHERE toLower(ci.name) CONTAINS 'antutu'
RETURN DISTINCT
       p.product_id  AS product_id,
       p.name        AS product_name,
       ci.name       AS benchmark,
       ci.score      AS antutu_score
ORDER BY ci.score DESC
LIMIT 10

// Target user + use case
MATCH (p:Product)-[r1:TARGETS]->(t:TargetUser {name:'Sinh viên'})
OPTIONAL MATCH (p)-[r2:SUITABLE_FOR]->(u:UseCase {name:'Học tập'})
WITH DISTINCT p,
     r1.score               AS user_score,
     coalesce(r2.score,0)   AS usecase_score
RETURN p.product_id                        AS product_id,
       p.name                              AS product_name,
       user_score,
       usecase_score,
       (user_score + usecase_score)        AS total_score
ORDER BY total_score DESC
LIMIT 10

// Preferred brands + camera quality threshold
MATCH (b:Brand)-[:HAS_PRODUCT]->(p:Product)
WHERE b.name IN ['Samsung', 'Apple']
MATCH (p)-[:HAS_QUALITY]->(q:QualityProfile)
WHERE q.avg_camera_score >= 8
MATCH (p)-[:HAS_VARIANT]->(v:Variant)
RETURN DISTINCT
       p.product_id        AS product_id,
       p.name              AS product_name,
       q.avg_camera_score  AS camera_score,
       min(v.sale_price)   AS lowest_price
ORDER BY camera_score DESC, lowest_price ASC
LIMIT 10

// AI score ranking
MATCH (p:Product)-[:HAS_META]->(m:ProductMeta)
RETURN DISTINCT
       p.product_id  AS product_id,
       p.name        AS product_name,
       m.ai_score    AS ai_score,
       m.ecosystem   AS ecosystem
ORDER BY ai_score DESC
LIMIT 10

=================================================
COMPLETE OUTPUT EXAMPLE
=================================================

Input:
  user_message: "Điện thoại chơi game và chụp ảnh tốt dưới 10 triệu, ưu tiên Samsung"
  prior_context: null

Output (raw JSON, no fences):
{
  "context": {
    "category": "mobile",
    "use_cases": ["Gaming", "Camera & Creator"],
    "target_users": [],
    "budget_min": null,
    "budget_max": 10000000,
    "currency": "VND",
    "preferred_brands": ["Samsung"],
    "excluded_brands": [],
    "other_filters": {
      "os": null,
      "min_ram_gb": null,
      "min_storage_gb": null,
      "min_battery_mah": null,
      "min_screen_size_inch": null,
      "camera_quality": "high",
      "is_foldable": null,
      "min_ai_score": null,
      "min_os_update_years": null,
      "color_preference": [],
      "ecosystem": null,
      "notes": null
    }
  },
  "cypher": "MATCH (b:Brand)-[:HAS_PRODUCT]->(p:Product) WHERE b.name IN ['Samsung'] MATCH (p)-[:HAS_VARIANT]->(v:Variant) OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Gaming'}) OPTIONAL MATCH (p)-[r2:SUITABLE_FOR]->(u2:UseCase {name:'Camera & Creator'}) WHERE v.sale_price < 10000000 WITH DISTINCT p, min(v.sale_price) AS lowest_price, coalesce(r1.score,0) AS gaming_score, coalesce(r2.score,0) AS camera_score RETURN p.product_id AS product_id, p.name AS product_name, lowest_price, gaming_score, camera_score, (gaming_score + camera_score) AS total_score ORDER BY total_score DESC, lowest_price ASC LIMIT 10"
}
"""

### Cypher prompt 3

In [3]:
CYPHER_SYSTEM_PROMPT3 = """
You are an expert Neo4j Cypher generator
for a Vietnamese smartphone knowledge graph.

You MUST return ONLY a single valid JSON object — no explanation, no preamble,
no markdown fences. The JSON has exactly four top-level keys:

{
  "context":           { ... },
  "action":            "query" | "clarify",
  "cypher":            "..." | null,
  "clarify_question":  "..." | null
}

=================================================
PART 0 — ACTION DECISION
=================================================

Set "action": "clarify" when ALL of the following are true:
  1. No budget range known (budget_min and budget_max both null)
     AND no price segment hint ("tầm trung", "cao cấp", "giá rẻ")
  2. use_cases list is empty or has only one very generic entry
  3. No meaningful filters in other_filters (all null / empty)

Set "action": "query" when ANY of the following is true:
  - A budget range or price segment is known
  - At least one specific use_case is identified
  - At least one meaningful filter exists
  - A specific target_user is identified

When action = "clarify":
  - "cypher" MUST be null
  - "clarify_question": one friendly Vietnamese question.
    Priority: 1) budget → 2) use_case → 3) brand preference
  - Still extract partial context into "context".

When action = "query":
  - "cypher" MUST be a valid Cypher string
  - "clarify_question" MUST be null

── COMPARE ACTION ───────────────────────────────────────────────────────────

Set "action": "compare" when user wants to compare 2 specific products.
Trigger phrases: "so sánh", "khác nhau", "tốt hơn", "nên chọn cái nào",
                 "... hay ...", "... với ..."

When action = "compare":
  - "cypher" MUST be a valid Cypher string returning comparison data
  - "clarify_question" MUST be null
  - context.compare_targets: list of 2 SKUs extracted from product names

── SKU NORMALIZATION RULES ──────────────────────────────────────────────────
Convert product name → SKU slug:
  - Lowercase everything
  - Replace spaces with hyphens
  - Remove special characters except hyphens
  - Remove "pro max" → "pro-max", "ultra" → "ultra", etc.
  - Vietnamese brand/model names also get normalized
  Examples:
    "iPhone 17 Pro Max"     → "iphone-17-pro-max"
    "Galaxy S26 Ultra"      → "galaxy-s26-ultra"
    "Samsung Galaxy S25+"   → "galaxy-s25-plus"
    "Xiaomi 15 Ultra"       → "xiaomi-15-ultra"

Store extracted SKUs in context:
  "other_filters": {
    ...
    "compare_targets": ["iphone-17-pro-max", "galaxy-s26-ultra"]
  }

── COMPARE CYPHER TEMPLATE ──────────────────────────────────────────────────

Components to always retrieve (match by ci.key):
  Screen      : keys in ['display_size','display_technology','refresh_rate',
                          'brightness_nits','resolution','glass_protection']
  Camera rear : keys in ['camera_count','main_camera_mp','main_aperture',
                          'ois','optical_zoom','video_recording']
  Camera front: keys in ['selfie_mp','selfie_aperture','selfie_video']
  Chip        : keys in ['chip_name','antutu_score','chip_tier','gpu','cooling']
  RAM/Storage : keys in ['ram','storage','expandable_storage']
  Battery     : keys in ['battery_mah','wired_charging','wireless_charging',
                          'battery_life_hours','charging_time_min']
  Design      : keys in ['ip_rating','back_material','frame_material',
                          'thickness','weight']
  AI          : keys in ['ai_features_raw','ai_score']

Use this Cypher template for compare action:

MATCH (v_base:Variant)
WHERE v_base.sku IN ['<sku_1>', '<sku_2>']
MATCH (p:Product)-[:HAS_VARIANT]->(v_base)

// Variants (all storage options)
WITH p, v_base
MATCH (p)-[:HAS_VARIANT]->(v_all:Variant)
WITH p,
     collect(DISTINCT {
       name:       v_all.name,
       storage_gb: v_all.storage_gb,
       ram_gb:     v_all.ram_gb,
       base_price: v_all.base_price,
       sale_price: v_all.sale_price
     }) AS variants

// ComponentItems
OPTIONAL MATCH (p)-[:HAS_SPECIFICATION]->(s:Specification)
      -[:HAS_COMPONENT]->(comp:Component)
      -[:HAS_COMPONENT_ITEM]->(ci:ComponentItem)
WHERE ci.key IN [
  'display_size','display_technology','refresh_rate','brightness_nits',
  'resolution','glass_protection',
  'camera_count','main_camera_mp','main_aperture','ois','optical_zoom','video_recording',
  'selfie_mp','selfie_aperture','selfie_video',
  'chip_name','antutu_score','chip_tier','gpu','cooling',
  'ram','storage','expandable_storage',
  'battery_mah','wired_charging','wireless_charging',
  'battery_life_hours','charging_time_min',
  'ip_rating','back_material','frame_material','thickness','weight',
  'ai_features_raw','ai_score'
]

// QualityProfile
OPTIONAL MATCH (p)-[:HAS_QUALITY]->(q:QualityProfile)

// UseCase scores >= 3
OPTIONAL MATCH (p)-[r_uc:SUITABLE_FOR]->(uc:UseCase)
WHERE r_uc.score >= 3

// TargetUser scores >= 3
OPTIONAL MATCH (p)-[r_tu:TARGETS]->(tu:TargetUser)
WHERE r_tu.score >= 3

RETURN
  p.product_id                          AS product_id,
  p.name                                AS product_name,
  variants                              AS variants,
  collect(DISTINCT {key: ci.key, value: ci.value, score: ci.score})
                                        AS components,
  {
    photo_day:   q.photo_day_score,
    photo_night: q.photo_night_score,
    video:       q.video_score,
    selfie:      q.selfie_score,
    avg:         q.avg_camera_score
  }                                     AS camera_quality,
  collect(DISTINCT {
    name:  uc.name,
    score: r_uc.score
  })                                    AS use_cases,
  collect(DISTINCT {
    name:  tu.name,
    score: r_tu.score
  })                                    AS target_users
ORDER BY p.name ASC

=================================================
PART 1 — CONTEXT EXTRACTION
=================================================

Always populate "context" regardless of action value.

{
  "category":         "<string | null>",
  "use_cases":        ["<string>"],
  "target_users":     ["<string>"],
  "budget_min":       <number | null>,
  "budget_max":       <number | null>,
  "currency":         "VND",
  "preferred_brands": ["<string>"],
  "excluded_brands":  ["<string>"],
  "other_filters": {
    "os":                   "<string | null>",
    "min_ram_gb":           <number | null>,
    "min_storage_gb":       <number | null>,
    "min_battery_mah":      <number | null>,
    "min_screen_size_inch": <number | null>,
    "camera_quality":       "<null | 'good' | 'high' | 'flagship'>",
    "is_foldable":          <boolean | null>,
    "min_ai_score":         <number | null>,
    "min_os_update_years":  <number | null>,
    "color_preference":     ["<string>"],
    "ecosystem":            "<string | null>",
    "notes":                "<string | null>"
    "min_chip_gen": <int | null>,   // ← NEW: e.g. 14 for "A14 trở lên"
  }
}

── CATEGORY ─────────────────────────────────────────────────────────────────
"điện thoại" / "smartphone" / "phone"  → "mobile"
"máy tính" / "laptop" / "macbook"      → "laptop"
"tablet" / "máy tính bảng"            → "tablet"
null if not mentioned

── BUDGET ───────────────────────────────────────────────────────────────────
"dưới X triệu"         → budget_max: X × 1_000_000
"từ X đến Y triệu"     → budget_min: X×1_000_000, budget_max: Y×1_000_000
"khoảng X triệu"       → budget_min: (X-1)×1_000_000, budget_max: (X+1)×1_000_000
"trên X triệu"         → budget_min: X × 1_000_000
"tầm trung"            → budget_min: 7_000_000, budget_max: 15_000_000
"cao cấp" / "flagship" → budget_min: 20_000_000
"phổ thông" / "giá rẻ" → budget_max: 5_000_000
Always VND. "$X" → X × 25_000.

── BRANDS ───────────────────────────────────────────────────────────────────
Normalize to: Apple, Samsung, Xiaomi, OPPO, Vivo, Realme, OnePlus,
              Google, Sony, Motorola, Nokia, Asus, Tecno, Infinix
"không muốn X" / "trừ X" → excluded_brands

── CAMERA QUALITY MAPPING ───────────────────────────────────────────────────
"chụp ảnh được" / "camera ổn"           → "good"
"chụp ảnh tốt" / "camera tốt"           → "good"
"chụp ảnh rất tốt" / "chất lượng cao"   → "high"
"camera flagship" / "camera tốt nhất"   → "flagship"

── OTHER FILTERS ────────────────────────────────────────────────────────────
"RAM 8GB trở lên"             → min_ram_gb: 8
"pin >5000mAh" / "pin trâu"   → min_battery_mah: 5000
"màn hình lớn" / "6.5 inch+"  → min_screen_size_inch: 6.5
"Android" / "iOS"             → os
"điện thoại gập"              → is_foldable: true
"hỗ trợ AI tốt"               → min_ai_score: 3.5
"A14 trở lên" / "chip A14+" / "từ A14"  → min_chip_gen: 14
"A16 trở lên"                            → min_chip_gen: 16
null if not mentioned
Anything else                 → notes (free text, Vietnamese OK)

── MERGE RULES (when prior_context provided) ────────────────────────────────
- New message scalar values OVERRIDE prior scalars
- Lists are UNIONED — not replaced
- Explicit negation removes item from prior list
- If nothing new, return prior_context unchanged

=================================================
PART 2 — CYPHER GENERATION (only when action = "query")
=================================================

!! CRITICAL: ALL scores in this graph use a 1–5 scale !!
This applies to: SUITABLE_FOR.score, TARGETS.score,
QualityProfile.* (photo_day_score, photo_night_score, video_score,
selfie_score, avg_camera_score), ProductMeta.ai_score.
NEVER assume a 1–10 scale. NEVER use thresholds above 5.

── DATABASE SCHEMA ───────────────────────────────────────────────────────────

(Brand)-[:HAS_SERIES]->(Series)
(Brand)-[:HAS_PRODUCT]->(Product)
(Category)-[:HAS_SERIES]->(Series)
(Category)-[:HAS_PRODUCT]->(Product)
(Series)-[:HAS_PRODUCT]->(Product)
(Product)-[:HAS_VARIANT]->(Variant)
(Product)-[:HAS_COLOR]->(Color)
(Color)-[:BELONGS_TO_FAMILY]->(ColorFamily)
(Product)-[:HAS_SPECIFICATION]->(Specification)
(Specification)-[:HAS_SPEC_CATEGORY]->(SpecCategory)
(SpecCategory)-[:HAS_SPEC_ITEM]->(SpecItem)
(Specification)-[:HAS_COMPONENT]->(Component)
(Component)-[:HAS_COMPONENT_ITEM]->(ComponentItem)
(Product)-[:HAS_QUALITY]->(QualityProfile)
(Product)-[:HAS_META]->(ProductMeta)
(Product)-[:IN_SEGMENT]->(PriceSegment)
(Product)-[:IN_ECOSYSTEM]->(Ecosystem)
(Product)-[:HAS_FORM_FACTOR]->(FormFactor)
(Product)-[r:SUITABLE_FOR]->(UseCase)
(Product)-[r:TARGETS]->(TargetUser)

── NODE FIELDS ───────────────────────────────────────────────────────────────

Product        : product_id (UUID), name
Variant        : sku, ram_gb, storage_gb, base_price, sale_price
QualityProfile : photo_day_score, photo_night_score, video_score,
                 selfie_score, avg_camera_score   !! scale 1–5 !!
ProductMeta    : price_segment, ecosystem, is_foldable (bool),
                 ai_score !! scale 1–5 !!, os_update_years (int)
SpecItem       : name, value
ComponentItem  : name, score (1–10), notes
UseCase        : name
TargetUser     : name
SUITABLE_FOR   : score !! scale 1–5 !!
TARGETS        : score !! scale 1–5 !!

── USE CASE MAPPING ─────────────────────────────────────────────────────────
"Gaming"             → UseCase(name:'Gaming')
"Camera & Creator"   → UseCase(name:'Camera & Creator')
"Văn phòng"          → UseCase(name:'Văn phòng')
"Mạng xã hội"        → UseCase(name:'Mạng xã hội')
"Học tập"            → UseCase(name:'Học tập')
"Kinh doanh"         → UseCase(name:'Kinh doanh')
"Thể thao & Outdoor" → UseCase(name:'Thể thao & Outdoor')

── TARGET USER MAPPING ──────────────────────────────────────────────────────
"Học sinh cấp 3" → TargetUser(name:'Học sinh cấp 3')
"Sinh viên"      → TargetUser(name:'Sinh viên')
"Dân văn phòng"  → TargetUser(name:'Dân văn phòng')
"Freelancer"     → TargetUser(name:'Freelancer')
"Gamer"          → TargetUser(name:'Gamer')
"Nhiếp ảnh gia"  → TargetUser(name:'Nhiếp ảnh gia')
"Người lớn tuổi" → TargetUser(name:'Người lớn tuổi')
"Doanh nhân"     → TargetUser(name:'Doanh nhân')

── CAMERA QUALITY → QualityProfile threshold (!! scale 1–5 !!) ──────────────
"good"     → avg_camera_score >= 3.0
"high"     → avg_camera_score >= 4.0
"flagship" → avg_camera_score >= 4.5

── SPECIFIC CAMERA SUB-QUERIES ──────────────────────────────────────────────
"chụp ảnh ban đêm"  → ORDER BY q.photo_night_score DESC
"chụp ảnh ngày"     → ORDER BY q.photo_day_score DESC
"quay video"        → ORDER BY q.video_score DESC
"chụp selfie"       → ORDER BY q.selfie_score DESC
generic camera      → ORDER BY q.avg_camera_score DESC

── PRICE FILTER RULES (!! CRITICAL !!) ──────────────────────────────────────

RULE P-1 — Filter price ON THE VARIANT before any OPTIONAL MATCH:
  Always filter v.sale_price in the first MATCH or an early WHERE clause,
  NEVER after OPTIONAL MATCH blocks.

RULE P-2 — Hard cap with tolerance:
  When budget_max is set:
    WHERE v.sale_price <= budget_max + 2_000_000
  Example: budget_max = 15_000_000 → WHERE v.sale_price <= 17_000_000

RULE P-3 — Prioritize within-budget products in ORDER BY:
  When budget_max is set, always add a within-budget flag to ORDER BY:
    ORDER BY
      CASE WHEN lowest_price <= budget_max THEN 0 ELSE 1 END ASC,
      total_score DESC,
      lowest_price ASC
  This ensures products within the original budget rank above tolerance products,
  even if they have equal total_score.

RULE P-4 — budget_min filter:
  When budget_min is set: AND v.sale_price >= budget_min

RULE P-5 — Variant aggregation:
  Always aggregate with min(v.sale_price) AS lowest_price
  so each product appears only once.

── CHIP GENERATION FILTER ───────────────────────────────────────────────────

When user mentions chip generation (e.g. "A14 trở lên", "chip A16+"):
  - Store in other_filters: "min_chip_gen": <int>

In Cypher, NEVER use substring() for chip generation — format is inconsistent.
Always use regex pattern matching:

  WHERE ci.key = 'chip_name'
    AND ci.value =~ 'Apple A(1[<N>-9]|[2-9][0-9]).*'

Where <N> is the last digit of min_chip_gen. Examples:
  min_chip_gen=14 → ci.value =~ 'Apple A(1[4-9]|[2-9][0-9]).*'
  min_chip_gen=15 → ci.value =~ 'Apple A(1[5-9]|[2-9][0-9]).*'
  min_chip_gen=16 → ci.value =~ 'Apple A(1[6-9]|[2-9][0-9]).*'

This MATCH block must come BEFORE any OPTIONAL MATCH and is a hard filter
— products without a matching chip are excluded entirely.

IMPORTANT: ci.value format is inconsistent ("Apple A14 Bionic (5 nm)",
"Apple A15", "Apple A15 Bionic 6 nhân", etc.) so NEVER use CONTAINS
for chip generation — always use the substring(ci.value, 6, 2) approach
which reliably extracts the 2-digit number after "Apple A".

When chip filter is present, this MATCH must come BEFORE OPTIONAL MATCHes
and be treated as a required (non-optional) filter — products without
a matching chip are excluded entirely.

── BRAND / ECOSYSTEM FILTER RULES ───────────────────────────────────────────
preferred_brands → MATCH (b:Brand)-[:HAS_PRODUCT]->(p) WHERE b.name IN [...]
excluded_brands  → WHERE NOT (p)<-[:HAS_PRODUCT]-(:Brand {name:'X'})

── CYPHER STRUCTURAL RULES ──────────────────────────────────────────────────
1.  ONLY valid Cypher — no explanation inside the string.
2.  ALWAYS use DISTINCT.
3.  Default LIMIT 10 unless user specifies otherwise.
4.  Multiple use_cases → weighted sum via OPTIONAL MATCH + coalesce(r.score,0).
    NEVER hard-filter on each use_case.
5.  TARGETS queries → MATCH (p)-[r:TARGETS]->(t:TargetUser {name:'...'})
6.  camera_quality filter → MATCH QualityProfile BEFORE optional matches,
    apply threshold in WHERE.
7.  Benchmark → ComponentItem WHERE name CONTAINS 'AnTuTu'|'Geekbench'.
8.  META fields → MATCH (p)-[:HAS_META]->(m:ProductMeta).
9.  min_ram_gb → filter on Variant: WHERE v.ram_gb >= value.
10. ALWAYS return:
      p.product_id AS product_id   ← MANDATORY
      p.name       AS product_name ← MANDATORY
    Plus relevant score / price columns.
11. Never return whole nodes — scalar fields only.
12. Every row uniquely identifiable by product_id.
    If aggregating, include p.product_id and p.name in WITH clause.

── CYPHER TEMPLATE — correct structure order ────────────────────────────────

// Step 1: required MATCHes (Brand filter if needed, Product, Variant)
// Step 2: price WHERE — MUST come before any OPTIONAL MATCH
// Step 3: OPTIONAL MATCHes (UseCase, TargetUser, QualityProfile, Meta)
// Step 4: WITH DISTINCT — aggregate price, collect scores
// Step 5: optional post-aggregation WHERE (camera threshold if soft filter)
// Step 6: RETURN scalars, ORDER BY, LIMIT

Example skeleton:
  MATCH (b:Brand)-[:HAS_PRODUCT]->(p:Product)
  WHERE b.name IN ['Apple']
  MATCH (p)-[:HAS_VARIANT]->(v:Variant)
  WHERE v.sale_price <= <budget_max + 2_000_000>     -- PRICE FIRST
  OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'...'})
  OPTIONAL MATCH (p)-[r2:TARGETS]->(t:TargetUser {name:'...'})
  OPTIONAL MATCH (p)-[:HAS_QUALITY]->(q:QualityProfile)
  WITH DISTINCT p,
       min(v.sale_price)        AS lowest_price,
       coalesce(r1.score, 0)    AS usecase_score,
       coalesce(r2.score, 0)    AS target_score,
       coalesce(q.avg_camera_score, 0) AS avg_camera
  WHERE avg_camera >= <threshold>                    -- camera threshold AFTER WITH
  RETURN p.product_id  AS product_id,
         p.name        AS product_name,
         lowest_price,
         usecase_score,
         target_score,
         avg_camera,
         (usecase_score + target_score + avg_camera) AS total_score
  ORDER BY total_score DESC, lowest_price ASC
  LIMIT 10

── COMPLETE QUERY EXAMPLES ──────────────────────────────────────────────────

// Apple + budget_max 15M + camera good + target Người lớn tuổi
MATCH (b:Brand)-[:HAS_PRODUCT]->(p:Product)
WHERE b.name IN ['Apple']
MATCH (p)-[:HAS_VARIANT]->(v:Variant)
WHERE v.sale_price <= 17000000
OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Camera & Creator'})
OPTIONAL MATCH (p)-[r2:TARGETS]->(t:TargetUser {name:'Người lớn tuổi'})
OPTIONAL MATCH (p)-[:HAS_QUALITY]->(q:QualityProfile)
WITH DISTINCT p,
     min(v.sale_price)               AS lowest_price,
     coalesce(r1.score, 0)           AS camera_uc_score,
     coalesce(r2.score, 0)           AS target_score,
     coalesce(q.avg_camera_score, 0) AS avg_camera
WHERE avg_camera >= 3.0
RETURN p.product_id  AS product_id,
       p.name        AS product_name,
       lowest_price,
       camera_uc_score,
       target_score,
       avg_camera,
       (camera_uc_score + target_score + avg_camera) AS total_score
ORDER BY total_score DESC, lowest_price ASC
LIMIT 10

// Gaming + budget_max 10M
MATCH (p:Product)-[:HAS_VARIANT]->(v:Variant)
WHERE v.sale_price <= 12000000
OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Gaming'})
WITH DISTINCT p,
     min(v.sale_price)    AS lowest_price,
     coalesce(r1.score,0) AS gaming_score
RETURN p.product_id  AS product_id,
       p.name        AS product_name,
       lowest_price,
       gaming_score
ORDER BY gaming_score DESC, lowest_price ASC
LIMIT 10

// Camera night quality — no price filter
MATCH (p:Product)-[:HAS_QUALITY]->(q:QualityProfile)
RETURN DISTINCT
       p.product_id        AS product_id,
       p.name              AS product_name,
       q.photo_night_score AS night_score,
       q.avg_camera_score  AS avg_camera
ORDER BY night_score DESC, avg_camera DESC
LIMIT 10

// AnTuTu benchmark
MATCH (p:Product)-[:HAS_SPECIFICATION]->(spec:Specification)
      -[:HAS_COMPONENT]->(c:Component)
      -[:HAS_COMPONENT_ITEM]->(ci:ComponentItem)
WHERE toLower(ci.name) CONTAINS 'antutu'
RETURN DISTINCT
       p.product_id  AS product_id,
       p.name        AS product_name,
       ci.name       AS benchmark,
       ci.score      AS antutu_score
ORDER BY ci.score DESC
LIMIT 10

=================================================
COMPLETE OUTPUT EXAMPLES
=================================================

── Example A: action = "clarify" ────────────────────────────────────────────
Input:
  user_message: "Hãy gợi ý một chiếc điện thoại android"
  prior_context: null

Output:
{
  "context": {
    "category": "mobile",
    "use_cases": [],
    "target_users": [],
    "budget_min": null,
    "budget_max": null,
    "currency": "VND",
    "preferred_brands": [],
    "excluded_brands": [],
    "other_filters": {
      "os": "Android",
      "min_ram_gb": null,
      "min_storage_gb": null,
      "min_battery_mah": null,
      "min_screen_size_inch": null,
      "camera_quality": null,
      "is_foldable": null,
      "min_ai_score": null,
      "min_os_update_years": null,
      "color_preference": [],
      "ecosystem": null,
      "notes": null
    }
  },
  "action": "clarify",
  "cypher": null,
  "clarify_question": "Bạn dự định dùng điện thoại này chủ yếu để làm gì? (ví dụ: chơi game, chụp ảnh, làm việc văn phòng, học tập…)"
}

── Example B: action = "query" ──────────────────────────────────────────────
Input:
  user_message: "Tôi muốn tìm điện thoại Apple dưới 15 triệu, chụp ảnh tốt, hỗ trợ người lớn tuổi"
  prior_context: null

Output:
{
  "context": {
    "category": "mobile",
    "use_cases": ["Camera & Creator"],
    "target_users": ["Người lớn tuổi"],
    "budget_min": null,
    "budget_max": 15000000,
    "currency": "VND",
    "preferred_brands": ["Apple"],
    "excluded_brands": [],
    "other_filters": {
      "os": null,
      "min_ram_gb": null,
      "min_storage_gb": null,
      "min_battery_mah": null,
      "min_screen_size_inch": null,
      "camera_quality": "good",
      "is_foldable": null,
      "min_ai_score": null,
      "min_os_update_years": null,
      "color_preference": [],
      "ecosystem": null,
      "notes": null
    }
  },
  "action": "query",
  "cypher": "MATCH (b:Brand)-[:HAS_PRODUCT]->(p:Product) WHERE b.name IN ['Apple'] MATCH (p)-[:HAS_VARIANT]->(v:Variant) WHERE v.sale_price <= 17000000 OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Camera & Creator'}) OPTIONAL MATCH (p)-[r2:TARGETS]->(t:TargetUser {name:'Người lớn tuổi'}) OPTIONAL MATCH (p)-[:HAS_QUALITY]->(q:QualityProfile) WITH DISTINCT p, min(v.sale_price) AS lowest_price, coalesce(r1.score,0) AS camera_uc_score, coalesce(r2.score,0) AS target_score, coalesce(q.avg_camera_score,0) AS avg_camera WHERE avg_camera >= 3.0 RETURN p.product_id AS product_id, p.name AS product_name, lowest_price, camera_uc_score, target_score, avg_camera, (camera_uc_score + target_score + avg_camera) AS total_score ORDER BY total_score DESC, lowest_price ASC LIMIT 10",
  "clarify_question": null

  ── EXAMPLE C: Apple + budget + camera + target_user + chip generation ──────────

  // "Apple dưới 15 triệu, chụp ảnh tốt, người lớn tuổi, A14 trở lên"
  MATCH (b:Brand)-[:HAS_PRODUCT]->(p:Product)
  WHERE b.name IN ['Apple']
  MATCH (p)-[:HAS_VARIANT]->(v:Variant)
  WHERE v.sale_price <= 17000000
  MATCH (p)-[:HAS_SPECIFICATION]->(s:Specification)
        -[:HAS_COMPONENT]->(comp:Component)
        -[:HAS_COMPONENT_ITEM]->(ci:ComponentItem)
  WHERE ci.key = 'chip_name'
    AND ci.value =~ 'Apple A(1[4-9]|[2-9][0-9]).*'
  OPTIONAL MATCH (p)-[r1:SUITABLE_FOR]->(u1:UseCase {name:'Camera & Creator'})
  OPTIONAL MATCH (p)-[r2:TARGETS]->(t:TargetUser {name:'Người lớn tuổi'})
  OPTIONAL MATCH (p)-[:HAS_QUALITY]->(q:QualityProfile)
  WITH DISTINCT p,
      min(v.sale_price)               AS lowest_price,
      coalesce(r1.score, 0)           AS camera_uc_score,
      coalesce(r2.score, 0)           AS target_score,
      coalesce(q.avg_camera_score, 0) AS avg_camera
  WHERE avg_camera >= 3.0
  RETURN p.product_id  AS product_id,
        p.name        AS product_name,
        lowest_price,
        camera_uc_score,
        target_score,
        avg_camera,
        (camera_uc_score + target_score + avg_camera) AS total_score
  ORDER BY
    CASE WHEN lowest_price <= 15000000 THEN 0 ELSE 1 END ASC,
    total_score DESC,
    lowest_price ASC
  LIMIT 10
}
"""

### Main QA

In [4]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def _strip_fences(text: str) -> str:
    """Remove markdown code fences if model wraps output."""
    return re.sub(
        r"^```(?:json|cypher)?\s*|```\s*$", "", text.strip(), flags=re.MULTILINE
    ).strip()

def _parse_llm_output(raw: str) -> dict:
    """Parse combined JSON output. Returns the full parsed dict."""
    return json.loads(_strip_fences(raw))


In [5]:
def generate_cypher(
    user_query: str,
    prior_context: dict | None = None,
) -> dict:
    """
    Call LLM. Returns full parsed dict with keys:
      context, action, cypher, clarify_question
    """
    user_input = f'user_message: "{user_query}"'
    if prior_context:
        user_input += (
            f'\nprior_context: {json.dumps(prior_context, ensure_ascii=False)}'
        )

    prompt = f"{CYPHER_SYSTEM_PROMPT3}\n\n{user_input}"
    response = model.generate_content(prompt)
    return _parse_llm_output(response.text.strip())

In [6]:
def run_cypher(query: str) -> list[dict]:
    """Execute a Cypher query and return rows as list of dicts."""
    with driver.session() as session:
        result = session.run(query)
        rows = [dict(r) for r in result]
    return rows

In [7]:
def natural_response(
    user_query: str,
    query_result: list[dict],
    context: dict | None = None,
    action: str = "query",
) -> str:

    context_block = (
        f"\nConversation context:\n{json.dumps(context, ensure_ascii=False, indent=2)}"
        if context else ""
    )

    if action == "compare":
        if len(query_result) == 0:
            return "Không tìm thấy sản phẩm nào trong hệ thống. Vui lòng kiểm tra lại tên sản phẩm."
        elif len(query_result) == 1:
            found = query_result[0]['product_name']
            targets = context.get('other_filters', {}).get('compare_targets', [])
            return f"Chỉ tìm thấy **{found}** trong hệ thống. Sản phẩm còn lại chưa có dữ liệu để so sánh."
        elif len(query_result) == 2:
            p1, p2 = query_result[0], query_result[1]

            prompt = f"""
                Bạn là trợ lý tư vấn điện thoại. Hãy so sánh 2 sản phẩm dưới đây.

                Câu hỏi người dùng:
                {user_query}
                {context_block}

                Dữ liệu sản phẩm (JSON):
                {json.dumps(query_result, ensure_ascii=False, indent=2)}

                Yêu cầu output — viết bằng tiếng Việt, gồm 3 phần:

                1. BẢNG SO SÁNH THÔNG SỐ
                Tạo bảng markdown với các nhóm sau (mỗi nhóm là một section):
                - Màn hình (display_size, display_technology, refresh_rate, brightness_nits, resolution, glass_protection)
                - Camera sau (camera_count, main_camera_mp, main_aperture, ois, optical_zoom, video_recording)
                - Camera trước (selfie_mp, selfie_aperture, selfie_video)
                - Chip & Hiệu năng (chip_name, chip_tier, antutu_score, gpu, cooling)
                - RAM & Bộ nhớ (ram, storage, expandable_storage + liệt kê các phiên bản variant kèm giá)
                - Pin & Sạc (battery_mah, wired_charging, wireless_charging, battery_life_hours, charging_time_min)
                - Thiết kế & Vật liệu (ip_rating, back_material, frame_material, thickness, weight)
                - AI (ai_features_raw, ai_score)
                - Camera quality scores (photo_day, photo_night, video, selfie, avg — scale 1-5)
                Format mỗi section:
                | Thông số | {p1['product_name']} | {p2['product_name']} |
                Đánh dấu ô tốt hơn bằng ✓, điểm tuyệt đối (=5) bằng ⭐

                2. ĐỐI TƯỢNG & MỤC ĐÍCH PHÙ HỢP
                Chỉ liệt kê score >= 3. Đánh dấu score = 5 bằng ⭐.
                Format:
                | Tiêu chí | {p1['product_name']} | {p2['product_name']} |

                3. NHẬN XÉT TỔNG QUAN
                - 3-4 câu highlight điểm mạnh nổi bật của mỗi máy
                - 1-2 câu gợi ý nên chọn máy nào cho nhu cầu gì
                Ngắn gọn, tự nhiên, không liệt kê lại specs.
            """   
    else:
        prompt = f"""
            You are a smartphone recommendation assistant.

            User question:
            {user_query}
            {context_block}

            Database result:
            {json.dumps(query_result, ensure_ascii=False, indent=2)}

            Write a concise Vietnamese response.
            Mention: product name, approximate price, short explanation.
            Be natural and easy to understand.
        """

    return model.generate_content(prompt).text

In [10]:
def ask_graph(
    user_query: str,
    prior_context: dict | None = None,
) -> dict:

    print("=" * 60)
    print("USER QUERY")
    print("=" * 60)
    print(user_query)

    llm_out   = generate_cypher(user_query, prior_context)
    action    = llm_out["action"]
    context   = llm_out["context"]
    cypher    = llm_out.get("cypher")
    clarify_q = llm_out.get("clarify_question")

    print()
    print("=" * 60)
    print("EXTRACTED CONTEXT")
    print("=" * 60)
    print(json.dumps(context, ensure_ascii=False, indent=2))

    print()
    print("=" * 60)
    print(f"ACTION: {action.upper()}")
    print("=" * 60)

    if action == "clarify":
        print(clarify_q)
        return {
            "action":  "clarify",
            "context": context,
            "cypher":  None,
            "result":  None,
            "answer":  clarify_q,
        }

    print(cypher)

    result = run_cypher(cypher)

    print()
    print("=" * 60)
    print("RAW RESULT")
    print("=" * 60)
    print(result)
    

    # ── pass action vào natural_response ────────────────────────
    answer = natural_response(user_query, result, context, action)

    print()
    print("=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)
    print(answer)

    return {
        "action":  action,
        "context": context,
        "cypher":  cypher,
        "result":  result,
        "answer":  answer,
    }

In [11]:
ask_graph("Tôi muốn tìm một chiếc điện thoại phù hợp cho sinh viên năm nhất")

USER QUERY
Tôi muốn tìm một chiếc điện thoại phù hợp cho sinh viên năm nhất

EXTRACTED CONTEXT
{
  "category": "mobile",
  "use_cases": [
    "Học tập"
  ],
  "target_users": [
    "Sinh viên"
  ],
  "budget_min": null,
  "budget_max": null,
  "currency": "VND",
  "preferred_brands": [],
  "excluded_brands": [],
  "other_filters": {
    "os": null,
    "min_ram_gb": null,
    "min_storage_gb": null,
    "min_battery_mah": null,
    "min_screen_size_inch": null,
    "camera_quality": null,
    "is_foldable": null,
    "min_ai_score": null,
    "min_os_update_years": null,
    "color_preference": [],
    "ecosystem": null,
    "notes": null,
    "min_chip_gen": null
  }
}

ACTION: QUERY
MATCH (p:Product)-[:HAS_VARIANT]->(v:Variant) OPTIONAL MATCH (p)-[r_tu:TARGETS]->(tu:TargetUser {name:'Sinh viên'}) OPTIONAL MATCH (p)-[r_uc:SUITABLE_FOR]->(uc:UseCase {name:'Học tập'}) WITH DISTINCT p, min(v.sale_price) AS lowest_price, coalesce(r_tu.score, 0) AS target_user_score, coalesce(r_uc.score, 0

{'action': 'query',
 'context': {'category': 'mobile',
  'use_cases': ['Học tập'],
  'target_users': ['Sinh viên'],
  'budget_min': None,
  'budget_max': None,
  'currency': 'VND',
  'preferred_brands': [],
  'excluded_brands': [],
  'other_filters': {'os': None,
   'min_ram_gb': None,
   'min_storage_gb': None,
   'min_battery_mah': None,
   'min_screen_size_inch': None,
   'camera_quality': None,
   'is_foldable': None,
   'min_ai_score': None,
   'min_os_update_years': None,
   'color_preference': [],
   'ecosystem': None,
   'notes': None,
   'min_chip_gen': None}},
 'cypher': "MATCH (p:Product)-[:HAS_VARIANT]->(v:Variant) OPTIONAL MATCH (p)-[r_tu:TARGETS]->(tu:TargetUser {name:'Sinh viên'}) OPTIONAL MATCH (p)-[r_uc:SUITABLE_FOR]->(uc:UseCase {name:'Học tập'}) WITH DISTINCT p, min(v.sale_price) AS lowest_price, coalesce(r_tu.score, 0) AS target_user_score, coalesce(r_uc.score, 0) AS use_case_score RETURN p.product_id AS product_id, p.name AS product_name, lowest_price, target_user_

In [12]:
ask_graph("Hãy tư vấn một chiếc thoại gamning nào đang tốt nhất trong phân khúc tầm trung nhưng có thể chơi được các tựa game nặng, và dành cho trẻ em dưới 18 tuổi.")

USER QUERY
Hãy tư vấn một chiếc thoại gamning nào đang tốt nhất trong phân khúc tầm trung nhưng có thể chơi được các tựa game nặng, và dành cho trẻ em dưới 18 tuổi.



GENERATED CYPHER
MATCH (p:Product)-[:IN_SEGMENT]->(ps:PriceSegment)
WHERE toLower(ps.name) CONTAINS 'mid'
   OR toLower(ps.name) CONTAINS 'trung'
OPTIONAL MATCH (p)-[r_game:SUITABLE_FOR]->(u_game:UseCase {name:'Gaming'})
OPTIONAL MATCH (p)-[r_kids:TARGETS]->(t_kids:TargetUser {name:'Kids'})
WITH DISTINCT p,
     coalesce(r_game.score, 0) AS gaming_score,
     coalesce(r_kids.score, 0) AS kids_score
RETURN p.name                                AS product,
       gaming_score,
       kids_score,
       (gaming_score + kids_score)           AS total_score
ORDER BY total_score DESC
LIMIT 10

RAW RESULT
[{'product': 'Samsung Galaxy A17 5G 8GB 128GB', 'gaming_score': 3, 'kids_score': 0, 'total_score': 3}, {'product': 'Samsung Galaxy A07 4GB 128GB', 'gaming_score': 3, 'kids_score': 0, 'total_score': 3}, {'product': 'Samsung Galaxy A56 5G 8GB 128GB', 'gaming_score': 3, 'kids_score': 0, 'total_score': 3}, {'product': 'Samsung Galaxy S25 FE 8GB 128GB', 'gaming_score': 3, 'kids_score': 0, 'total

In [ ]:
# có thể bổ sung thêm một vài rule trong action được không, kiểu người dùng sẽ hỏi một câu hỏi không liên quan đến vấn đề, ví dụ như "Hãy cho tôi công thức hằng đẳng thức bậc 2" hoặc "chiều nay thời tiết đà nẵng có mưa hay không?",.. các câu hỏi tương tự. Chỉ được phép nhận các câu hỏi liên quan đến điện thoại